# Fig 4A – average predicted cell-type contribution per dataset batch

Mean nu-SVR fraction of each major cell type (MCT) per batch; all other cell types are summed into *Others*. Also prints the MCT numbers quoted in the Results text.

Loading code: `deconv_common.py`.

In [ ]:
# {dataset}: Flomics_2, chalasani_merged, chen, ibarra, moufarrej, reggiardo, roskams, tao, toden, zhu, decru
FRACTIONS_FILE = "/mnt/efs/home/gasole/projects/FL-cfRNAmeta/script/2026_10_26_scripts_review/inputs_test/deconv_results{dataset}/dec_cfRNAmeta_fractions_results.csv"
SAMPLE_LIST_FILE = "/mnt/efs/home/gasole/projects/FL-cfRNAmeta/script/2026_10_26_scripts_review/inputs_test/deconv_results{dataset}/sample_list.txt"

OUTPUT_DIR = "../../output/deconvolution"

In [ ]:
import sys
from pathlib import Path

import pandas as pd

sys.path[:0] = [".", ".."]
import deconv_common

OUT = Path(OUTPUT_DIR)
deconv_common.set_fonts()

In [ ]:
metadata = deconv_common.load_metadata()
labels = deconv_common.load_labels()
per_dataset, cell_types = deconv_common.load_deconvolution(metadata, FRACTIONS_FILE, SAMPLE_LIST_FILE)
mcts = deconv_common.select_mcts(per_dataset, cell_types)
print("major cell types:", mcts)
data = deconv_common.add_plot_batches(pd.concat(per_dataset.values()), labels)

## Numbers quoted in the Results

In [ ]:
mct_sum = data[mcts].sum(axis=1) * 100
ep_sum = data[["erythrocyte", "platelet"]].sum(axis=1) * 100
print(f"MCTs: {mct_sum.mean():.2f} +/- {mct_sum.std():.2f} %")
print(f"erythrocyte + platelet: {ep_sum.mean():.2f} +/- {ep_sum.std():.2f} %")

## Plot

In [ ]:
import barplot_stacked_plot

names = {"erythrocyte": "Erythrocyte", "macrophage": "Macrophage", "neutrophil": "Neutrophil",
         "b cell": "B cell", "platelet": "Platelet", "monocyte": "Monocyte"}
assert set(names) == set(mcts), f"MCTs changed: {mcts}"
plot_df = data.rename(columns=names)
value_columns = [names.get(c, c) for c in cell_types]
colors = {names.get(k, k): v for k, v in deconv_common.CELL_TYPE_COLORS.items()} | {"others": "gray"}

df1, df2, ax, artists = barplot_stacked_plot.create_stacked_barplot(
    dataset=plot_df, meta_column="dataset_batch_custom", meta_order=deconv_common.BATCH_ORDER,
    value_columns=value_columns, value_order="median_descending",
    output=str(OUT / "fig4a_celltype_barplot"), dpi=600,
    scaling="mean", focus_value=list(names.values()), color_map=colors,
    others_label="Others",
    add_category_border=True, category_border_width=0.25,
    group_by_column="dataset_short_name", show_group_label=False, group_label_fontsize=8,
    group_label_y_offset=-0.065, group_bracket_linewidth=0.25,
    group_bracket_vertical_line_length=0.01, group_bracket_horizontal_line_length=0.4,
    group_position="middle", add_connecting_shades=False,
    ylabel="Predicted cell type fraction", xlabel_fontsize=5, ylabel_fontsize=5,
    x_tick_label_fontsize=5, x_tick_label_rotation=45, x_ticks_label_pad=-0.03,
    yticks=[0, 0.2, 0.4, 0.6, 0.8, 1], y_tick_label_fontsize=5,
    show_xlabel=False, show_title=False, fig_width=4.4, fig_height=2,
    hide_top_spine=True, hide_right_spine=True,
    legend_title="Cell type", legend_position="custom bottom",
    legend_x_pos=0.0, legend_y_pos=-0.62, legend_title_fontsize=5, legend_fontsize=5,
    boxes_column="broad_protocol_category_short", boxes_color_map=deconv_common.BPC_COLORS,
    boxes_y_position=-0.04, boxes_height=0.03, boxes_width=0.95, boxes_legend=True,
    hide_bottom_spine=True, hide_left_spine=True, hide_bottom_tick=True, hide_left_tick=True,
    boxes_borderwidth=0, boxes_legend_pos="bottom", boxes_legend_x_pos=0.0,
    boxes_legend_y_pos=-0.75, boxes_legend_title="Broad Protocol Category (BPC)",
    boxes_legend_fontsize=5, ax_width=1.01 * 2.3, aspect=1.8)